In [1]:

import glob
import pandas as pd
import os
import pyarrow as pa
import numpy as np
import pyarrow.parquet as pq
import re


from scipy.stats import spearmanr, pearsonr


In [2]:
from load_experiment_data import (
    train_dataset_name,
    test_dataset_name,
    train_dataset_split,
    test_dataset_split,
    load_data_and_estimators,
    explanation_types,
    explanation_m,
    linear_coders,
    explanation_k,
    explanation_lambda,
    explanation_seed
    )

/root/.local/lib/python3.10/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/root/.local/lib/python3.10/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `ty

In [3]:
NUM_TEST_INSTANCES = 1000 

In [4]:
k = len(explanation_k)
m = 1 # len(explanation_m)
lambda_ = len(explanation_lambda)
explanation_seed_ = 5

In [28]:
explanation_types

[explanations.TopKMostInfluential,
 explanations.TopKLeastInfluential,
 explanations.TopKMostHelpful,
 explanations.TopKMostHarmful,
 explanations.FacilityLocationMostHarmful,
 explanations.FacilityLocationMostHelpful,
 explanations.FacilityLocationMostInfluential,
 explanations.FacilityLocationLeastInfluential,
 explanations.DIVINEMostHelpful,
 explanations.DIVINEMostHarmful,
 explanations.DIVINEMostInfluential,
 explanations.DIVINELeastInfluential,
 explanations.AIDE]

In [5]:
num_explanation_types = 1 # Self

num_explanation_types += k*m # explanations.TopKMostInfluential,
num_explanation_types += k*m # explanations.TopKLeastInfluential,
num_explanation_types += k*m # explanations.TopKMostHelpful,
num_explanation_types += k*m # explanations.TopKMostHarmful,
num_explanation_types += k*m*lambda_ # explanations.FacilityLocationMostHarmful,
num_explanation_types += k*m*lambda_ # explanations.FacilityLocationMostHelpful,
num_explanation_types += k*m*lambda_ # explanations.FacilityLocationMostInfluential,
num_explanation_types += k*m*lambda_ # explanations.FacilityLocationLeastInfluential,
num_explanation_types += k*m # explanations.DIVINEMostHelpful,
num_explanation_types += k*m # explanations.DIVINEMostHarmful,
num_explanation_types += k*m # explanations.DIVINEMostInfluential,
num_explanation_types += k*m # explanations.DIVINELeastInfluential,
num_explanation_types += k*m # explanations.AIDE
num_explanation_types += k*m*explanation_seed_
num_explanation_types

137

In [42]:
num_explanation_types_bm25 = 1 # Self
num_explanation_types_bm25 += k*m # explanations.TopKMostInfluential,
num_explanation_types_bm25 += k*m # explanations.TopKLeastInfluential,
num_explanation_types_bm25 += k*m*lambda_ # explanations.FacilityLocationMostInfluential,
num_explanation_types_bm25 += k*m*lambda_ # explanations.FacilityLocationLeastInfluential,
num_explanation_types_bm25 += k*m # explanations.DIVINEMostInfluential,
num_explanation_types_bm25 += k*m # explanations.DIVINELeastInfluential,
num_explanation_types_bm25 += k*m # explanations.AIDE
num_explanation_types_bm25 += k*m*explanation_seed_
num_explanation_types_bm25

81

## Scoring

In [ ]:
df_scoring = pq.ParquetDataset("results/scoring").read().to_pandas()

In [ ]:
group_sizes = df_scoring.groupby(
    ["model","explanation_type","train_dataset","train_split","test_dataset","test_split",
     "estimator", "linear_coder"]
).size()

group_sizes[group_sizes != NUM_TEST_INSTANCES]

model                                                        explanation_type                                                                                         train_dataset                                 train_split  test_dataset                                  test_split  estimator                                   linear_coder                     
Llama-3.2-1B_tulu-3-sft-olmo-2-mixture-0225_lr0.0001_seed42  25 by facility location from Top-100 most influential (scores with largest absolute value). lambda=0.25  loris3/tulu-3-sft-olmo-2-mixture-0225-sample  train        loris3/tulu-3-sft-olmo-2-mixture-0225-sample  test        DataInfEstimator: fast_implementation=True  MSECoderProjUSimpSparseSoftThresh    999
dtype: int64

In [ ]:
n_settings = num_explanation_types*3*2 + num_explanation_types_bm25*3


In [ ]:
len(df_scoring["explanation_type"].unique()) == num_explanation_types

True

In [ ]:
assert n_settings == len(df_scoring.groupby(["explanation_type", "model", "estimator"]))
assert len(group_sizes[group_sizes != NUM_TEST_INSTANCES]) == 0

6389999

## Validation

In [23]:
df_validation = pq.ParquetDataset("results/validation").read().to_pandas()


In [24]:
len(df_validation)

901172

In [ ]:
set(df_scoring["explanation_type"].unique()) - set(df_validation["explanation_type"].unique())

In [53]:
missing_validation_explanation_types, missing_validation_models, missing_validation_estimators = zip(*set(
    df_scoring[["explanation_type", "model", "estimator"]]
    .drop_duplicates()
    .itertuples(index=False, name=None)
) - set(
    df_validation[["explanation_type", "model", "estimator"]]
    .drop_duplicates()
    .itertuples(index=False, name=None)
))
missing_validation_explanation_types = set(missing_validation_explanation_types)
missing_validation_models = set(missing_validation_models)
missing_validation_estimators = set(missing_validation_estimators)
print(missing_validation_explanation_types)
# print(missing_validation_models)
# print(missing_validation_estimators)

{'5 random examples with seed 42', '10 by AIDE from Top-100.', '1 random examples with seed 10', '10 random examples with seed 10', '1 random examples with seed 3', '25 random examples with seed 42', '10 by facility location from Top-100 least influential (scores closest to zero). lambda=0.0', '25 random examples with seed 9', 'The test instance (as a sanity check)', '25 by facility location from Top-100 least influential (scores closest to zero). lambda=0.75', '1 random examples with seed 42', '1 by AIDE from Top-100.', '10 random examples with seed 42', '10 by facility location from Top-100 most influential (scores with largest absolute value). lambda=0.25', '5 random examples with seed 6', '1 random examples with seed 6', '25 random examples with seed 10', '5 random examples with seed 9', '1 random examples with seed 9', '10 random examples with seed 6', '5 random examples with seed 10', '10 random examples with seed 3', '10 by facility location from Top-100 least influential (score

In [25]:
len(df_validation["explanation_type"].unique()) == num_explanation_types_validation

True

In [26]:
group_sizes = df_validation.groupby(
    ["model","explanation_type", "train_dataset","train_split","test_dataset","test_split",
     "estimator"]
).size()

group_sizes[group_sizes != NUM_TEST_INSTANCES]

model                                                          explanation_type                                                                            train_dataset                                 train_split  test_dataset                                  test_split  estimator                                 
Llama-3.2-1B_tulu-3-sft-olmo-2-mixture-0225_lr0.0001_seed42    Top-1 most harmful (most positive scores)                                                   loris3/tulu-3-sft-olmo-2-mixture-0225-sample  train        loris3/tulu-3-sft-olmo-2-mixture-0225-sample  test        LESSEstimator: normalize=True                 122
OLMo-2-0425-1B_tulu-3-sft-olmo-2-mixture-0225_lr0.0001_seed42  5 by AIDE from Top-100.                                                                     loris3/tulu-3-sft-olmo-2-mixture-0225-sample  train        loris3/tulu-3-sft-olmo-2-mixture-0225-sample  test        DataInfEstimator: fast_implementation=True    945
Qwen2.5-0.5B_tulu-3-sft-olmo-2-mixture-02

In [44]:
 len(df_validation.groupby(["explanation_type", "model", "estimator"]))

905

In [45]:
n_settings = num_explanation_types*3*2 + num_explanation_types_bm25*3
assert n_settings == len(df_validation.groupby(["explanation_type", "model", "estimator"]))

AssertionError: 